# PFE ML — Train V1 (one-shot launcher)

A clean launcher for the V1 continuity-risk model. It pulls the latest code, runs `app/tools/train_continuity_model.py` once, and shows the results — no hunting through the phase notebooks.

One run produces (in `ml-artifacts/runs/<run_name>/`): the model + an archived copy, the full metric set (AP, AUC, **Gini, KS, Brier, log loss, ECE, MCC, bootstrap CIs**), **isotonic calibration** (raw vs calibrated), **SHAP** global explanations, 12+ charts, and a self-documenting `OUTPUTS.md`. New-style runs are tagged `output_schema_version 2.0`.

**Runtime:** pick **CPU · High-RAM** (Runtime → Change runtime type). HGB is CPU-only; the dataset load is the bottleneck, so RAM matters more than GPU. Only use a GPU if you set `MODEL_FAMILY` to `catboost`/`xgboost` and add `--gpu`.

> The training script lives in the **cloned repo** (`/content/pfein/back_end/app/tools/`), not on your Drive. It's run as a module (`python -m app.tools.train_continuity_model`), which is why you won't find it in the Drive file browser.

In [ ]:
import os, sys, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

DUCKDB_TMP.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)
os.chdir(BACKEND_DIR)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'collabs/requirements-colab.txt'])

# Confirm we actually have the new code (calibration + SHAP) on the pulled branch.
script = BACKEND_DIR / 'app' / 'tools' / 'train_continuity_model.py'
text = script.read_text(encoding='utf-8')
head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'log', '--oneline', '-1']).decode().strip()
print('HEAD commit :', head)
print('Script found:', script.exists())
print('Has calibration + SHAP:', ('OUTPUT_SCHEMA_VERSION' in text) and ('_write_shap_artifacts' in text))
if 'OUTPUT_SCHEMA_VERSION' not in text:
    raise SystemExit('Old code on the clone. Re-run after: !rm -rf /content/pfein  then re-run this cell.')

## 1. Configure the run

Edit these knobs. Defaults reproduce the standard 2M-row, time-split-2024 HGB run with calibration + SHAP on.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'

TARGET            = 'continuity_risk_12m_label'
# One family or a list. Set to ['hgb'] for a single fast run, or to all four
# for a head-to-head shootout. Each run lands in ml-artifacts/runs/<family>/...
MODEL_FAMILIES    = ['hgb', 'lightgbm', 'catboost', 'xgboost']
TRAIN_START_YEAR  = 2017
TRAIN_END_YEAR    = 2024
# 2M is plenty for a family comparison and keeps the 4-family shootout under
# ~15 min total. Pick the winner here, then retrain it at 20M for the final.
MAX_ROWS          = 2_000_000
CALIBRATE         = True
RUN_SHAP          = True
PARAMS_FILE       = ''             # only applied when MODEL_FAMILIES has one entry
GPU               = False          # only helps catboost / xgboost
# Optional short tag appended to the run folder name so feature-set generations
# are visually distinguishable (e.g. 'v3-traj' for the trajectory-features build).
# Leave empty to let the trainer auto-detect feature_schema_version from the
# features parquet manifest -- runs trained on the 3.0-trajectory-sector parquet
# will land under runs/<family>/...feat-3-0-trajectory-sector/ automatically.
RUN_TAG           = ''

ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DATA_LAKE     = f'{DRIVE_ROOT}/data-lake'
print('Artifacts ->', ARTIFACTS_DIR)
print('Data lake ->', DATA_LAKE)
print(f'families={MODEL_FAMILIES}  years {TRAIN_START_YEAR}-{TRAIN_END_YEAR}  max_rows={MAX_ROWS}  calibrate={CALIBRATE}  shap={RUN_SHAP}  run_tag={RUN_TAG or "(auto)"}')

## 2. Check the data lake is present

The script reads features + labels parquet from Drive. This confirms they exist before a long run.

In [ ]:
import duckdb

features_root = Path(DATA_LAKE) / 'features' / 'company_year_features'
labels_root   = Path(DATA_LAKE) / 'features' / 'risk_labels'
if not list(features_root.rglob('*.parquet')):
    raise SystemExit(f'No feature parquet under {features_root}')
if not list(labels_root.rglob('*.parquet')):
    raise SystemExit(f'No label parquet under {labels_root}')

feat_glob = (features_root / '**' / '*.parquet').as_posix().replace("'", "''")
lab_glob  = (labels_root / '**' / '*.parquet').as_posix().replace("'", "''")
con = duckdb.connect()
try:
    counts = con.execute(
        f"SELECT (SELECT count(*) FROM read_parquet('{feat_glob}', union_by_name=true)) AS feature_rows, "
        f"(SELECT count(*) FROM read_parquet('{lab_glob}', union_by_name=true)) AS label_rows"
    ).df()
finally:
    con.close()
print(counts.to_string(index=False))

## 3. Train (one shot)

Builds the `python -m app.tools.train_continuity_model` command from your config and runs it. Expect ~10–20 min for HGB at 2M rows; calibration is cheap, SHAP adds ~3–5 min, bootstrap CIs a couple more.

In [ ]:
import shlex, time, json

results = {}
for family in MODEL_FAMILIES:
    print('=' * 70)
    print(f'Training {family}')
    print('=' * 70)
    cmd = [
        sys.executable, '-u', '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', DATA_LAKE,
        '--artifacts-dir', ARTIFACTS_DIR,
        '--target', TARGET,
        '--model-family', family,
        '--train-start-year', str(TRAIN_START_YEAR),
        '--train-end-year', str(TRAIN_END_YEAR),
        '--min-rows', '1000',
    ]
    if MAX_ROWS:
        cmd += ['--max-rows', str(MAX_ROWS)]
    cmd += ['--calibrate'] if CALIBRATE else ['--no-calibrate']
    cmd += ['--shap'] if RUN_SHAP else ['--no-shap']
    if RUN_TAG:
        cmd += ['--run-tag', RUN_TAG]
    if PARAMS_FILE and len(MODEL_FAMILIES) == 1:
        cmd += ['--params-file', PARAMS_FILE]
    if GPU:
        cmd += ['--gpu']

    print(' '.join(shlex.quote(part) for part in cmd), '\n')
    start = time.time()
    subprocess.run(cmd, check=True, cwd=str(BACKEND_DIR))
    elapsed = time.time() - start
    # The training script overwrites model_metadata.json (root) on every run,
    # so snapshot it now before the next family overwrites it.
    fam_meta = json.loads((Path(ARTIFACTS_DIR) / 'model_metadata.json').read_text(encoding='utf-8'))
    results[family] = {
        'run_name': fam_meta.get('run_name'),
        'run_dir': fam_meta.get('run_artifacts_dir'),
        'elapsed_seconds': elapsed,
        'metrics': fam_meta.get('metrics', {}),
    }
    print(f'Done in {elapsed:.0f}s  | run_dir = {results[family]["run_dir"]}\n')

print(f'All {len(MODEL_FAMILIES)} families trained.')

## 4. Headline results

Reads `model_metadata.json` (just written by this run) and prints the metric set, including the new credit-risk + calibration numbers and bootstrap CIs.

In [ ]:
import json

meta = json.loads((Path(ARTIFACTS_DIR) / 'model_metadata.json').read_text(encoding='utf-8'))
m = meta.get('metrics', {})

print('run_name        :', meta.get('run_name'))
print('output_schema   :', meta.get('output_schema_version'))
print('model_family    :', meta.get('model_family'))
print('split / years   :', meta.get('split_strategy'), '|', meta.get('train_start_year'), '->', meta.get('train_end_year'))
print('calibrated      :', meta.get('calibrated'), '(fit rows:', meta.get('calibration_fit_rows'), ')')
print('run folder      :', meta.get('run_artifacts_dir'))

def show(label, key, fmt='{:.4f}'):
    v = m.get(key)
    print(f'  {label:24s}: ' + (fmt.format(v) if isinstance(v, (int, float)) else str(v)))

print('\nDiscrimination (ranking):')
for lab, k in [('ROC AUC', 'roc_auc'), ('Average precision', 'average_precision'), ('Gini', 'gini'), ('KS statistic', 'ks_statistic')]:
    show(lab, k)
print(f"  AUC 95% CI              : {m.get('roc_auc_ci95_low')} - {m.get('roc_auc_ci95_high')}")
print(f"  AP  95% CI              : {m.get('average_precision_ci95_low')} - {m.get('average_precision_ci95_high')}")

print('\nProbability quality (raw -> calibrated):')
print(f"  Brier                   : {m.get('brier_score')} -> {m.get('calibrated_brier_score')}")
print(f"  ECE                     : {m.get('expected_calibration_error')} -> {m.get('calibrated_expected_calibration_error')}")
show('Log loss', 'log_loss')

print('\nThreshold 0.5:')
for lab, k in [('Precision', 'precision_at_0_5'), ('Recall', 'recall_at_0_5'), ('F1', 'f1_at_0_5'),
               ('MCC', 'mcc'), ('Balanced accuracy', 'balanced_accuracy'), ('Specificity', 'specificity_at_0_5')]:
    show(lab, k)

## 5. Run summary, outputs manifest, and charts

Renders `run_summary.md` and `OUTPUTS.md` from this run's folder, then displays the key charts inline.

In [ ]:
from IPython.display import Markdown, Image, display

run_dir = Path(meta['run_artifacts_dir'])

for doc in ('run_summary.md', 'OUTPUTS.md'):
    path = run_dir / doc
    if path.exists():
        display(Markdown(f'### {doc}'))
        display(Markdown(path.read_text(encoding='utf-8')))

charts = [
    'headline_metrics.png', 'calibration_curve.png', 'cumulative_gains_curve.png',
    'precision_recall_curve.png', 'roc_curve.png',
    'shap_summary_bar.png', 'shap_summary_beeswarm.png', 'top_feature_importances.png',
]
display(Markdown('### Charts'))
for name in charts:
    path = run_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print(name, '-> not produced (see OUTPUTS.md / shap_error.txt if SHAP)')

## 6. Family shootout comparison

Only meaningful when `MODEL_FAMILIES` has more than one entry. Builds a side-by-side AP / AUC / Gini / KS / calibrated Brier+ECE / top-K table across the families trained in this notebook run, sorts by AP, and identifies the winner. The CSV is saved so it can drop straight into your rapport.

In [ ]:
import pandas as pd
from datetime import datetime

if len(results) <= 1:
    print(f'Only {len(results)} family trained — nothing to compare. Set MODEL_FAMILIES to a list of multiple families to use this cell.')
else:
    rows = []
    for family, info in results.items():
        m = info['metrics']
        rows.append({
            'family':              family,
            'run_dir':             Path(info['run_dir']).name,
            'training_seconds':    round(info['elapsed_seconds']),
            'roc_auc':             m.get('roc_auc'),
            'average_precision':   m.get('average_precision'),
            'gini':                m.get('gini'),
            'ks_statistic':        m.get('ks_statistic'),
            'brier_calibrated':    m.get('calibrated_brier_score'),
            'ece_calibrated':      m.get('calibrated_expected_calibration_error'),
            'f1_at_0_5':           m.get('f1_at_0_5'),
            'top_0.1pct_precision': next((seg.get('precision') for seg in m.get('top_k_analysis', []) if seg.get('segment') == 'top_0.1%'), None),
            'top_1pct_lift':       next((seg.get('lift')      for seg in m.get('top_k_analysis', []) if seg.get('segment') == 'top_1.0%'), None),
            'conformal_calibration_source': m.get('conformal_calibration_source'),
        })
    shootout = pd.DataFrame(rows).sort_values('average_precision', ascending=False).reset_index(drop=True)
    print('Family shootout (sorted by average precision):\n')
    display(shootout)

    shootout_csv = Path(ARTIFACTS_DIR) / f'shootout_{datetime.now().strftime("%Y%m%d-%H%M%S")}.csv'
    shootout.to_csv(shootout_csv, index=False)
    print(f'\nSaved comparison to: {shootout_csv}')
    winner = shootout.iloc[0]
    print(f'\nWinner by AP: {winner["family"]}  (AP={winner["average_precision"]:.4f}, AUC={winner["roc_auc"]:.4f}).')
    print('To deploy the winner at 20M, set MODEL_FAMILIES = [winner], MAX_ROWS = 20_000_000, and re-run.')

## 7. Promote the winner to the deployable location

Copies the chosen family's `model.joblib` + isotonic + conformal calibrators + metadata up to the root `ml-artifacts/` (the path the backend loader and `publish_prediction_results.py` read from). After this cell, the root `model.joblib` is the canonical deployable.

Default: **LightGBM** — ties HGB on every metric (ΔAP 0.0008, inside the 95% CI) and trains 23% faster. Change `PROMOTE_FAMILY` to `'hgb'` or `'auto'` (AP winner) if you prefer.

In [ ]:
import shutil

# Which family to promote to the root ml-artifacts/ (the deployable location).
# 'auto' picks the AP winner from the shootout; otherwise set a specific family
# name. Default is 'lightgbm' because it ties HGB on every metric (delta AP
# 0.0008, inside the 95% CI) and trains 23% faster -- a clean operational win.
PROMOTE_FAMILY = 'lightgbm'

if PROMOTE_FAMILY == 'auto':
    promote = shootout.iloc[0]['family']
    why = f'auto-selected AP winner (AP = {shootout.iloc[0]["average_precision"]:.4f})'
else:
    promote = PROMOTE_FAMILY
    why = 'manually selected'

if promote not in results:
    raise SystemExit(f'{promote!r} was not trained in this session. results keys = {list(results.keys())}')

run_dir = Path(results[promote]['run_dir'])
print(f'Promoting {promote}  ({why})')
print(f'Source run dir : {run_dir}')
print(f'Target dir     : {ARTIFACTS_DIR}\n')

# (source filename in the run dir, destination filename at the root)
copies = [
    ('model.joblib',                'model.joblib'),
    ('isotonic_calibrator.joblib',  'isotonic_calibrator.joblib'),
    ('conformal_calibrator.joblib', 'conformal_calibrator.joblib'),
    ('metadata.json',               'model_metadata.json'),  # root convention
]
for source_name, target_name in copies:
    src = run_dir / source_name
    if not src.exists():
        print(f'  --  {source_name} not present in run dir, skipping')
        continue
    dst = Path(ARTIFACTS_DIR) / target_name
    shutil.copy2(src, dst)
    print(f'  OK  {source_name}  ->  {dst.name}')

print('\nRoot ml-artifacts now reflects this run.')
print('To deploy in the backend: copy these files into back_end/app/ml/artifacts/.')
print('To publish to Mongo end-to-end: run app.tools.publish_prediction_results.')

## Where everything landed & next steps

- **This run:** `ml-artifacts/runs/<run_name>/` — model + archived copy, `isotonic_calibrator.joblib`, all CSVs, charts, `OUTPUTS.md`, tagged `output_schema_version 2.0`.
- **Live/deployable:** root `ml-artifacts/model.joblib` + `model_metadata.json` (overwritten by this run). To serve via the backend, copy `model.joblib` into `back_end/app/ml/artifacts/`.
- **Cross-run:** `model_run_index.jsonl` / `model_run_comparison.csv` now carry Gini, KS, Brier, ECE for every new run.

Tips:
- **Deploy the *best* run, not the latest:** every run now archives its own `model.joblib`, so use `pfe_ml_model_interrogation.ipynb` (it auto-resolves the best archived run) instead of assuming the root file is best.
- **Pure (uncalibrated) run:** set `CALIBRATE = False` — the model then trains on 100% of the rows; comparable to your older runs.
- **Full dataset:** set `MAX_ROWS = 0` and use a High-RAM runtime.
- **Verify the artifact** any time with `pfe_ml_v1_artifact_check.ipynb`.